In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import urllib.request
import time
import json
import os



class WeightQuantizeSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, weight, eps=1e-5):
        mean = weight.mean()
        weight_centered = weight - mean
        gamma = weight_centered.abs().mean()
        weight_scaled = weight_centered / (gamma + eps) 
        weight_clipped = torch.clamp(weight_scaled, min=-1.0, max=1.0)
        weight_quantized = torch.round(weight_clipped)
        return weight_quantized

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output, None

class ActivationQuantizeSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, eps=1e-5):
        Qb = 127.0
        abs_max = x.abs().max()
        scale = Qb / (abs_max + eps)
        x_scaled = x * scale
        x_quantized = torch.round(torch.clamp(x_scaled, -Qb, Qb))
        x_dequantized = x_quantized / scale
        return x_dequantized

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output, None

class BitLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=True, eps=1e-5):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.eps = eps
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * (1.0 / in_features**0.5))
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features))
        else:
            self.register_parameter('bias', None)

    def forward(self, x):
        quantized_x = ActivationQuantizeSTE.apply(x, self.eps)
        quantized_w = WeightQuantizeSTE.apply(self.weight, self.eps)
        out = F.linear(quantized_x, quantized_w, self.bias)
        return out


batch_size = 32
block_size = 8
max_iters = 3000
eval_interval = 300
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
n_embd = 32
n_head = 4
n_layer = 3

print(f"--- Running 1-Bit Model on device: {device} ---")

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "shakespeare.txt")
with open('shakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split):
    data_source = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_source) - block_size, (batch_size,))
    x = torch.stack([data_source[i:i+block_size] for i in ix])
    y = torch.stack([data_source[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(200)
        for k in range(200):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out



class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        # SWAP 1: nn.Linear -> BitLinear
        self.key = BitLinear(n_embd, head_size, bias=False) 
        self.query = BitLinear(n_embd, head_size, bias=False)
        self.value = BitLinear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k, q = self.key(x), self.query(x)
        wei = q @ k.transpose(-2, -1) * (C ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        return wei @ self.value(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        # SWAP 2: nn.Linear -> BitLinear (default bias=True)
        self.proj = BitLinear(n_embd, n_embd, bias=True) 

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.proj(out)

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            # SWAP 3: nn.Linear -> BitLinear
            BitLinear(n_embd, 4 * n_embd, bias=True),
            nn.ReLU(),
            BitLinear(4 * n_embd, n_embd, bias=True),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class BitGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        # SWAP 4: Final LM Head
        self.lm_head = BitLinear(n_embd, vocab_size, bias=True)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


model = BitGPT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("--- Starting Training (1.58-bit Weights) ---")
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

torch.save(model.state_dict(), 'tiny_gpt_1bit.pth')
print("--- Training Complete! Model saved as 'tiny_gpt_1bit.pth' ---\n")


print("--- Calculating 1-Bit PyTorch Metrics ---")
total_params = sum(p.numel() for p in model.parameters())
file_size_mb = os.path.getsize('tiny_gpt_1bit.pth') / (1024 * 1024)

model.eval()
context = torch.zeros((1, 1), dtype=torch.long, device=device)

# Peak VRAM
torch.cuda.reset_peak_memory_stats(device)
_ = model.generate(context, max_new_tokens=10)
peak_memory_mb = torch.cuda.max_memory_allocated(device) / (1024 * 1024)

# Speed (Tokens/Sec) - Warmup included
for _ in range(2):
    _ = model.generate(context, max_new_tokens=50)
torch.cuda.synchronize()

tokens_to_generate = 500
start_time = time.time()
_ = model.generate(context, max_new_tokens=tokens_to_generate)
torch.cuda.synchronize()
time_taken = time.time() - start_time
tokens_per_second = tokens_to_generate / time_taken

bit_metrics = {
    "model_type": "BitNet_1.58b_PyTorch_Simulation",
    "total_parameters": total_params,
    "file_size_mb": round(file_size_mb, 2),
    "peak_vram_mb": round(peak_memory_mb, 2),
    "generation_time_seconds": round(time_taken, 2),
    "tokens_per_second": round(tokens_per_second, 2)
}

print("\n=== 1-BIT PYTORCH RESULTS ===")
for key, value in bit_metrics.items():
    print(f"{key}: {value}")

with open("1bit_baseline_metrics.json", "w") as f:
    json.dump(bit_metrics, f, indent=4)

--- Running 1-Bit Model on device: cuda ---
--- Starting Training (1.58-bit Weights) ---
step 0: train loss 11.0661, val loss 11.0930
step 300: train loss 3.2640, val loss 3.2916
step 600: train loss 2.9354, val loss 2.9414
step 900: train loss 2.8263, val loss 2.8181
step 1200: train loss 2.7119, val loss 2.7130
step 1500: train loss 2.7256, val loss 2.7427
step 1800: train loss 2.5558, val loss 2.5581
step 2100: train loss 2.5312, val loss 2.5447
step 2400: train loss 2.5116, val loss 2.4956
step 2700: train loss 2.4930, val loss 2.4832
step 2999: train loss 2.4491, val loss 2.4521
--- Training Complete! Model saved as 'tiny_gpt_1bit.pth' ---

--- Calculating 1-Bit PyTorch Metrics ---

=== 1-BIT PYTORCH RESULTS ===
model_type: BitNet_1.58b_PyTorch_Simulation
total_parameters: 42369
file_size_mb: 0.19
peak_vram_mb: 18.53
generation_time_seconds: 9.04
tokens_per_second: 55.33
